In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df = pd.read_csv('../data/raw/transactions.csv')

In [3]:
df.head()

,transaction_id,customer_id,kyc_verified,account_age_days,transaction_amount,channel,timestamp,is_fraud
0,TXN_200000,CUST_799,Yes,1050,256369,Mobile,2025-08-12 02:10:24,0
1,TXN_200001,CUST_484,Yes,295,6581,Mobile,2025-08-25 01:14:31,0
2,TXN_200002,CUST_791,Yes,2083,4492,Mobile,2025-08-17 12:12:40,0
3,TXN_200003,CUST_664,Yes,2789,275413,POS,2025-08-07 06:23:54,0
4,TXN_200004,CUST_157,Yes,694,98098,POS,2025-08-20 21:55:54,0


In [4]:
df.shape

(5000, 8)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   transaction_id      5000 non-null   object
 1   customer_id         5000 non-null   object
 2   kyc_verified        5000 non-null   object
 3   account_age_days    5000 non-null   int64 
 4   transaction_amount  5000 non-null   int64 
 5   channel             5000 non-null   object
 6   timestamp           5000 non-null   object
 7   is_fraud            5000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 312.6+ KB


In [6]:
df.describe()

,account_age_days,transaction_amount,is_fraud
count,5000.000000,5000.000000,5000.000000
mean,1498.906400,39283.321000,0.086400
std,867.615096,70330.374967,0.280982
min,10.000000,1054.000000,0.000000
25%,732.000000,6000.500000,0.000000
50%,1481.500000,9670.000000,0.000000
75%,2254.250000,20324.750000,0.000000
max,2999.000000,299992.000000,1.000000


In [7]:
df.isnull().sum()

transaction_id        0
customer_id           0
kyc_verified          0
account_age_days      0
transaction_amount    0
channel               0
timestamp             0
is_fraud              0
dtype: int64

In [8]:
df.fillna({"kyc_verified": "No"}, inplace=True)

In [9]:
df.dropna(subset=["transaction_amount"], inplace=True)

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df.drop_duplicates(subset="transaction_id", inplace=True)

In [12]:
# Save processed data
df.to_csv('../data/processed/transactions_raw_cleaned.csv', index=False)

In [13]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [14]:
df = pd.read_csv('../data/processed/transactions_raw_cleaned.csv')

In [15]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

In [16]:
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
df['weekday'] = df['timestamp'].dt.weekday

In [17]:
df['kyc_verified'] = df['kyc_verified'].map({'Yes': 1, 'No': 0})

In [18]:
df.columns = df.columns.str.strip().str.lower() 

print(df['channel'].unique())

df = pd.get_dummies(df, columns=['channel'], drop_first=False)

['Mobile' 'POS' 'ATM' 'Web']


In [19]:
df.columns = df.columns.str.strip().str.lower()

df[['channel_atm', 'channel_mobile', 'channel_pos', 'channel_web']] = \
    df[['channel_atm', 'channel_mobile', 'channel_pos', 'channel_web']].astype(int)


In [20]:
cap_value = df['transaction_amount'].quantile(0.99)
df['transaction_amount'] = np.where(
    df['transaction_amount'] > cap_value,
    cap_value,
    df['transaction_amount']
)


## Feature Engineering


In [21]:
# Average transaction amount per customer
df['avg_txn_per_customer'] = df.groupby('customer_id')['transaction_amount'].transform('mean')

In [22]:
# Transaction count per customer
df['txns_count_per_customer'] = df.groupby('customer_id')['transaction_amount'].transform('count')

In [23]:
# Deviation from normal spending
df['amt_deviation'] = df['transaction_amount'] - df['avg_txn_per_customer']


In [24]:
# Big suspicious transactions (top 5%)
high_amount_threshold = df['transaction_amount'].quantile(0.95)
df['high_amount_flag'] = (df['transaction_amount'] > high_amount_threshold).astype(int)

In [25]:
# Night transaction flag
df['is_night'] = df['hour'].apply(lambda x: 1 if x >= 22 or x <= 5 else 0)

In [26]:
# Weekend transaction flag
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x in ["Saturday", "Sunday"] else 0)

In [27]:
scaler = MinMaxScaler()
cols_to_scale = ['transaction_amount', 'account_age_days']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

In [28]:
final_df = df.drop(columns=['customer_id'])

In [29]:
df.to_csv("../data/processed/transactions_processed.csv", index=False)

In [30]:
X = final_df.drop(columns=['is_fraud', 'transaction_id', 'timestamp'])
y = final_df['is_fraud']


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [32]:
train_df = X_train.copy()
train_df['is_fraud'] = y_train

In [33]:
test_df = X_test.copy()
test_df['is_fraud'] = y_test

In [34]:
train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)
